In [ ]:
# classift=y activity using ML-NOT TO BE USED ... JUST TRIED
# import pandas as pd

df = pd.read_csv("../OpenActive_Merged.csv")
print(df.shape)

(31780, 23)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

train_df = df.dropna(subset=["activity_raw", "activity_type"]).copy()
train_df = train_df[train_df["activity_type"] != "Other / Unspecified"]

print(train_df.shape)
print(train_df["activity_type"].value_counts())

(24657, 23)
activity_type
Personal Training             6988
Swimming                      5890
Racquet Sports                2524
Team Sports                   2320
Gym / Fitness                 2131
Bootcamp & Outdoor Fitness    1680
Yoga, Pilates & Studio        1192
Group Exercise                 974
Cycling                        632
Martial Arts                   120
Walking / Running              118
Dance                           80
Wellness & Facility Access       8
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    train_df["activity_raw"], train_df["activity_type"],
    test_size=0.2, random_state=42, stratify=train_df["activity_type"]
)

print(len(X_train), len(X_test))

19725 4932


In [7]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=2000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [8]:
preds = clf.predict(X_test_vec)
print(classification_report(y_test, preds))

                            precision    recall  f1-score   support

Bootcamp & Outdoor Fitness       0.97      1.00      0.99       336
                   Cycling       0.99      0.98      0.98       126
                     Dance       0.25      0.88      0.39        16
            Group Exercise       0.85      0.97      0.91       195
             Gym / Fitness       0.97      0.94      0.95       426
              Martial Arts       0.78      0.88      0.82        24
         Personal Training       0.99      0.96      0.97      1398
            Racquet Sports       1.00      1.00      1.00       505
                  Swimming       1.00      0.99      0.99      1178
               Team Sports       1.00      1.00      1.00       464
         Walking / Running       1.00      0.96      0.98        24
Wellness & Facility Access       0.08      0.50      0.14         2
    Yoga, Pilates & Studio       0.98      0.91      0.94       238

                  accuracy                    

In [9]:
other_rows = df[df["activity_type"] == "Other / Unspecified"].dropna(subset=["activity_raw"])
other_vec = vectorizer.transform(other_rows["activity_raw"])
predictions = clf.predict(other_vec)
probabilities = clf.predict_proba(other_vec).max(axis=1)

other_rows = other_rows.copy()
other_rows["predicted_type"] = predictions
other_rows["confidence"] = probabilities

reliable = other_rows[
    (other_rows["confidence"] > 0.7) &
    (~other_rows["predicted_type"].isin(["Dance", "Wellness & Facility Access"]))
]

# manually inspect a sample - does the raw text actually match the predicted category?
print(reliable[["activity_raw","predicted_type","confidence"]].sample(20, random_state=1))

           activity_raw     predicted_type  confidence
29363           Running  Walking / Running    0.926704
1743          Soft Play  Personal Training    0.922196
1742          Soft Play  Personal Training    0.922196
29370           Running  Walking / Running    0.926704
31676   Swimming Castle           Swimming    0.750797
1741          Soft Play  Personal Training    0.922196
29352           Running  Walking / Running    0.926704
29386           Running  Walking / Running    0.926704
29366           Running  Walking / Running    0.926704
29355           Running  Walking / Running    0.926704
31304   Poplar Group Ex       Martial Arts    0.752251
29381           Running  Walking / Running    0.926704
31420  Swimming Peckham           Swimming    0.824271
29349           Running  Walking / Running    0.926704
1632          Soft Play  Personal Training    0.922196
29388           Running  Walking / Running    0.926704
1710          Soft Play  Personal Training    0.922196
29368     

In [10]:
##NEW
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

gap = pd.read_csv("../GAP_data/gap_scores_equity.csv")
profile = pd.read_csv("../GAP_data/borough_profile.csv")

data = gap.merge(profile[["borough","top_venue_share","avg_price","pct_better","blended_avg_price","pct_free"]], on="borough", how="left")

features = ["inactive_shrunk", "sessions", "venues", "equity_gap", "top_venue_share", "blended_avg_price", "pct_better"]
X = data[features].fillna(0)

print(X.describe())

       inactive_shrunk      sessions    venues  equity_gap  top_venue_share  \
count        32.000000     32.000000  32.00000   32.000000        32.000000   
mean         23.618966   1866.468750   7.12500    5.355417        41.855387   
std           2.379521   2713.436352   3.85001    3.838126        20.765425   
min          19.821987      0.000000   0.00000    0.878808         0.000000   
25%          21.757878    153.500000   5.00000    2.723188        28.431033   
50%          23.839141    557.000000   7.00000    4.475481        41.945611   
75%          25.356921   2941.750000   8.00000    6.931829        47.661119   
max          30.014407  12838.000000  18.00000   20.645574        91.525424   

       blended_avg_price  pct_better  
count          32.000000   32.000000  
mean           10.001622   30.520657  
std            10.033945   39.112355  
min             0.000000    0.000000  
25%             6.129288    0.000000  
50%             8.834856    5.520131  
75%            

In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# elbow method - check inertia across different cluster counts
inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

for k, i in zip(range(2,8), inertias):
    print(k, round(i,1))

c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Wi

2 177.5
3 147.7
4 124.3
5 107.2
6 87.8
7 73.7


In [12]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

km = KMeans(n_clusters=4, random_state=42, n_init=10)
data["cluster"] = km.fit_predict(X_scaled)

print(data.groupby("cluster")[features].mean())
print()
print(data.groupby("cluster")["borough"].apply(list))

c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


         inactive_shrunk  sessions  venues  equity_gap  top_venue_share  \
cluster                                                                   
0              25.005461   3240.25   8.875    4.364586        38.330354   
1              22.370796    152.30   5.700    4.448723        37.949302   
2              22.901276    251.00   8.000    3.672258        41.434263   
3              21.822064   1221.80   4.200   10.676099        68.697960   

         blended_avg_price  pct_better  
cluster                                 
0                 9.442337    6.641034  
1                 9.025101   80.809597  
2                57.503077    0.000000  
3                 6.132553   13.789907  

cluster
0    [Barking and Dagenham, Bexley, Brent, Ealing, ...
1    [Camden, Croydon, Enfield, Greenwich, Hackney,...
2                                           [Haringey]
3    [Barnet, Bromley, Kensington and Chelsea, King...
Name: borough, dtype: object


In [13]:
pd.set_option("display.max_colwidth", None)
print(data.groupby("cluster")["borough"].apply(list))

cluster
0    [Barking and Dagenham, Bexley, Brent, Ealing, Harrow, Havering, Hillingdon, Hounslow, Newham, Redbridge, Southwark, Sutton, Tower Hamlets, Waltham Forest, Wandsworth, Westminster]
1                                                                           [Camden, Croydon, Enfield, Greenwich, Hackney, Hammersmith and Fulham, Islington, Lambeth, Lewisham, Merton]
2                                                                                                                                                                             [Haringey]
3                                                                                                  [Barnet, Bromley, Kensington and Chelsea, Kingston upon Thames, Richmond upon Thames]
Name: borough, dtype: object


In [14]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
pd.set_option("display.max_colwidth", None)

for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    data[f"cluster_k{k}"] = km.fit_predict(X_scaled)
    print(f"\n=== k = {k} ===")
    print(data.groupby(f"cluster_k{k}")[features].mean().round(2))
    print(data.groupby(f"cluster_k{k}")["borough"].apply(list))

c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



=== k = 3 ===
            inactive_shrunk  sessions  venues  equity_gap  top_venue_share  \
cluster_k3                                                                   
0                     22.43   5401.33   12.00        8.17            27.85   
1                     22.25    178.92    5.15        5.38            48.20   
2                     25.53   1922.54    6.85        4.03            45.47   

            blended_avg_price  pct_better  
cluster_k3                                 
0                        5.82       12.46  
1                        7.66       66.35  
2                       15.45        3.28  
cluster_k3
0                                                                           [Bexley, Kingston upon Thames, Southwark, Waltham Forest, Wandsworth, Westminster]
1          [Barnet, Bromley, Camden, Croydon, Enfield, Greenwich, Hackney, Hammersmith and Fulham, Islington, Lambeth, Lewisham, Merton, Richmond upon Thames]
2    [Barking and Dagenham, Brent, Ealing, Ha

c:\Users\Hp\anaconda3\envs\text_analytics\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [15]:
data["cluster"] = data["cluster_k3"]
cluster_names = {
    0: "broad reliable supply",
    1: "data-quality limited (Better-dependent)",
    2: "genuine high need"
}
data["cluster_label"] = data["cluster"].map(cluster_names)

data.to_csv("../GAP_data/gap_scores_clustered.csv", index=False)
print(data[["borough","class","cluster_label"]].sort_values("cluster_label"))

                   borough                        class  \
31             Westminster  well-served (moderate need)   
2                   Bexley              emerging desert   
29          Waltham Forest  well-served (moderate need)   
19    Kingston upon Thames                  well-served   
30              Wandsworth                  well-served   
26               Southwark                  well-served   
20                 Lambeth         low need, low supply   
17               Islington         low need, low supply   
25    Richmond upon Thames         low need, low supply   
21                Lewisham         low need, low supply   
11  Hammersmith and Fulham         low need, low supply   
9                Greenwich              blind spot risk   
8                  Enfield                   blind spot   
6                  Croydon              blind spot risk   
5                   Camden         low need, low supply   
4                  Bromley         low need, low supply 

In [16]:
#why clusetring differs
mismatch = data[
    ((data["class"].isin(["genuine desert","blind spot","under-monitored","emerging desert"])) & (data["cluster_label"] != "genuine high need")) |
    ((data["class"].isin(["well-served","well-served (moderate need)"])) & (data["cluster_label"] == "genuine high need"))
]
print(mismatch[["borough","class","cluster_label"]])

    borough                        class  \
2    Bexley              emerging desert   
8   Enfield                   blind spot   
27   Sutton  well-served (moderate need)   

                              cluster_label  
2                     broad reliable supply  
8   data-quality limited (Better-dependent)  
27                        genuine high need  


In [17]:
print(data.groupby("cluster_label")["equity_gap"].agg(["mean","std","count"]))

                                             mean       std  count
cluster_label                                                     
broad reliable supply                    8.166943  6.632687      6
data-quality limited (Better-dependent)  5.379810  3.224890     13
genuine high need                        4.033397  1.865243     13


In [18]:
print(data[data["cluster_label"]=="broad reliable supply"][["borough","equity_gap"]])

                 borough  equity_gap
2                 Bexley    9.526453
19  Kingston upon Thames   20.645574
26             Southwark    2.645970
29        Waltham Forest    3.528406
30            Wandsworth    7.675749
31           Westminster    4.979506


In [1]:
'''de06gra*h5c feat4re 50*rtance'''
import pandas as pd

pop = pd.read_csv("../poplprofile_merged.csv")

# exclude aggregate/non-demographic rows if any (geography_level should be 'borough')
model_data = pop[pop["geography_level"] == "borough"].copy()
model_data = model_data.dropna(subset=["pct_inactive"])

print(model_data.shape)
print(model_data["demographic_variable"].value_counts())

(11589, 12)
demographic_variable
IMD10            1857
Age9             1793
Eth7             1424
Educ6            1247
Relig7           1218
NSSEC5            992
Disab3            757
Orient4           666
Gend3             614
ChildAgeU13       460
Maternity_pop     330
LondInOut         231
Name: count, dtype: int64


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

features = ["borough", "demographic_variable", "demographic_category", "survey_year"]
X = model_data[features].astype(str)
y = model_data["pct_inactive"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), features)
])
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=300, random_state=42, max_depth=12))
])
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("R²:", r2_score(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))

R²: 0.1986734217903514
MAE: 10.329638242430024


In [3]:
importances = model.named_steps["rf"].feature_importances_
feature_names = model.named_steps["prep"].get_feature_names_out()

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df["group"] = imp_df["feature"].str.extract(r"cat__(\w+)_")

grouped_importance = imp_df.groupby("group")["importance"].sum().sort_values(ascending=False)
print(grouped_importance)

group
demographic_category              0.594771
borough                           0.268104
survey_year                       0.078198
demographic_variable              0.058927
demographic_variable_Maternity    0.000000
Name: importance, dtype: float64


In [4]:
# rebuild features more cleanly - keep demographic_variable and demographic_category as separate,
# clean columns rather than relying on regex-splitting one-hot names afterward
features = ["borough", "demographic_variable", "survey_year"]
X = model_data[features].astype(str)
y = model_data["pct_inactive"]

# fit separate small models per demographic_variable to see which specific variable predicts best
for var in model_data["demographic_variable"].unique():
    subset = model_data[model_data["demographic_variable"] == var]
    if subset["demographic_category"].nunique() < 2 or len(subset) < 30:
        continue
    Xs = pd.get_dummies(subset[["borough","demographic_category"]].astype(str))
    ys = subset["pct_inactive"]
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=42)
    m = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=8)
    m.fit(Xtr, ytr)
    r2 = r2_score(yte, m.predict(Xte))
    print(f"{var}: R² = {r2:.3f}, n = {len(subset)}")

Age9: R² = 0.508, n = 1793
ChildAgeU13: R² = 0.403, n = 460
Disab3: R² = 0.569, n = 757
Educ6: R² = 0.334, n = 1247
Eth7: R² = 0.140, n = 1424
Gend3: R² = -0.089, n = 614
IMD10: R² = 0.023, n = 1857
LondInOut: R² = 0.619, n = 231
Maternity_pop: R² = -0.001, n = 330
NSSEC5: R² = 0.623, n = 992
Orient4: R² = -0.370, n = 666
Relig7: R² = 0.022, n = 1218


In [5]:
disab = pop[pop["demographic_variable"] == "Disab3"]
disab_summary = disab.groupby(["borough","demographic_category"])["pct_inactive"].mean().reset_index()
print(disab_summary[disab_summary["borough"].isin(["Kingston upon Thames","Bexley"])])

nssec = pop[pop["demographic_variable"] == "NSSEC5"]
nssec_summary = nssec.groupby(["borough","demographic_category"])["pct_inactive"].mean().reset_index()
print(nssec_summary[nssec_summary["borough"].isin(["Kingston upon Thames","Bexley"])])

                 borough        demographic_category  pct_inactive
8                 Bexley         Limiting disability     38.225079
9                 Bexley               No disability     22.379661
10                Bexley     Non-limiting disability     23.175281
11                Bexley  Not asked / Not applicable     26.416597
80  Kingston upon Thames         Limiting disability     28.269577
81  Kingston upon Thames               No disability     16.106165
82  Kingston upon Thames     Non-limiting disability     14.621746
83  Kingston upon Thames  Not asked / Not applicable     26.864688
                  borough                         demographic_category  \
12                 Bexley                              Aged <16 or 75+   
13                 Bexley             NS SEC 1-2: Higher social groups   
14                 Bexley             NS SEC 3-5: Middle social groups   
15                 Bexley              NS SEC 6-8: Lower social groups   
16                 Bexley  

In [6]:
import pandas as pd

data = pd.read_csv("../GAP_data/gap_scores_clustered.csv")

# normalize each component 0-1, then weight them into a transparent priority score
def norm(s): return (s - s.min()) / (s.max() - s.min())

data["priority_score"] = (
    0.35 * norm(data["inactive_shrunk"]) +      # overall need
    0.35 * norm(data["equity_gap"]) +            # hidden inequality
    0.30 * (1 - norm(data["sessions"]))          # supply scarcity (inverted)
)

print(data[["borough","class","priority_score"]].sort_values("priority_score", ascending=False).head(10))

                 borough                 class  priority_score
0   Barking and Dagenham       under-monitored        0.632174
2                 Bexley       emerging desert        0.604545
16              Hounslow       emerging desert        0.600185
1                 Barnet               average        0.596856
9              Greenwich       blind spot risk        0.594338
19  Kingston upon Thames           well-served        0.592363
24             Redbridge            blind spot        0.587795
8                Enfield            blind spot        0.583458
15            Hillingdon       emerging desert        0.569268
4                Bromley  low need, low supply        0.517801
